Install packages


In [13]:
# Install packages with compatible versions to avoid dependency conflicts
%pip install --no-deps geopandas
%pip install --no-deps "shapely>=2.0"
%pip install --no-deps pyproj
%pip install --no-deps rtree
%pip install --no-deps fiona
%pip install --no-deps pyogrio
%pip install --no-deps folium
# Install numpy with a compatible version
%pip install "numpy>=1.21,<2.0"
# Install pandas (should be compatible with the numpy version above)
%pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note

In [14]:
# Test imports to verify all packages are working
try:
    import geopandas as gpd
    import shapely
    import pyproj
    import rtree
    import fiona
    import pyogrio
    import folium
    import numpy as np
    import pandas as pd

    print("✓ All packages imported successfully!")
    print(f"GeoPandas version: {gpd.__version__}")
    print(f"Shapely version: {shapely.__version__}")
    print(f"NumPy version: {np.__version__}")
    print(f"Pandas version: {pd.__version__}")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Some packages may need to be installed manually.")

✓ All packages imported successfully!
GeoPandas version: 1.0.1
Shapely version: 2.0.7
NumPy version: 1.26.4
Pandas version: 2.0.3


In [ ]:
import os, sys

BASE = os.path.abspath(".")
DATA_DIR = os.path.join(BASE, "dataset")

CAD_PATH = os.path.join(DATA_DIR, "cadastre.gpkg")
ROADS_PATH = os.path.join(DATA_DIR, "roads.gpkg")
GNAF_PATH = os.path.join(DATA_DIR, "gnaf_prop.parquet")
# Optional, unused here:
TX_PATH = os.path.join(DATA_DIR, "transactions.parquet")

for p in [DATA_DIR, CAD_PATH, ROADS_PATH, GNAF_PATH]:
    assert os.path.exists(p), f"Missing: {p}"
print("✓ Found dataset folder and required files.")

✓ Found dataset folder and required files.


In [ ]:
import os, glob, sys

print("CWD:", os.getcwd())
print("dataset exists:", os.path.isdir("./dataset"))
print("dataset contents:", glob.glob("./dataset/*"))

CAD_PATH = "./dataset/cadastre.gpkg"
ROADS_PATH = "./dataset/roads.gpkg"
GNAF_PATH = "./dataset/gnaf_prop.parquet"

for p in [CAD_PATH, ROADS_PATH, GNAF_PATH]:
    print(p, "exists:", os.path.exists(p))

CWD: /Users/louistran/Desktop/task 2
dataset exists: True
dataset contents: ['./dataset/cadastre.gpkg', './dataset/roads.gpkg', './dataset/transactions.parquet', './dataset/gnaf_prop.parquet']
./dataset/cadastre.gpkg exists: True
./dataset/roads.gpkg exists: True
./dataset/gnaf_prop.parquet exists: True


In [7]:
import math, warnings, numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
import fiona

warnings.filterwarnings("ignore", category=UserWarning)


def pick_layer(gpkg_path, hints):
    layers = fiona.listlayers(gpkg_path)
    for h in hints:
        for L in layers:
            if h.lower() in L.lower():
                return L
    return layers[0]


def ensure_geometry_from_lonlat(df):
    lon_cols = [
        c for c in df.columns if c.lower() in ("lon", "lng", "longitude", "x", "long")
    ]
    lat_cols = [c for c in df.columns if c.lower() in ("lat", "latitude", "y")]
    if lon_cols and lat_cols:
        lon, lat = lon_cols[0], lat_cols[0]
        return gpd.GeoDataFrame(
            df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326"
        )
    if "geometry" in df.columns:
        return gpd.GeoDataFrame(df, geometry="geometry")
    raise ValueError("No lon/lat or geometry found in gnaf_prop.parquet.")


def to_local_metric_crs(gdf):
    gdf4326 = gdf.to_crs(4326)
    lon = float(gdf4326.geometry.x.mean())
    zone = int((lon + 180) // 6) + 1
    epsg = 32700 + zone if gdf4326.geometry.y.mean() < 0 else 32600 + zone
    return gdf.to_crs(epsg), epsg


def segmentize_polygon_edges(poly: Polygon):
    if poly is None or poly.is_empty or poly.exterior is None:
        return []
    coords = list(poly.exterior.coords)
    return [LineString([coords[i], coords[i + 1]]) for i in range(len(coords) - 1)]


def bearing_deg(p_from: Point, p_to: Point) -> float:
    dx, dy = p_to.x - p_from.x, p_to.y - p_from.y
    return (90 - math.degrees(math.atan2(dy, dx))) % 360  # 0°=N, 90°=E


def compass8(b: float) -> str:
    dirs = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    return dirs[int((b + 22.5) // 45) % 8]

Load from dataset


In [ ]:
CAD_HINTS = ["parcel", "cad", "lot", "property", "cadastre"]
ROAD_HINTS = ["road", "street", "transport", "centerline", "roads"]

CAD_LAYER = pick_layer(CAD_PATH, CAD_HINTS)
ROAD_LAYER = pick_layer(ROADS_PATH, ROAD_HINTS)
print("Using layers:", CAD_LAYER, "/", ROAD_LAYER)

cad = gpd.read_file(CAD_PATH, layer=CAD_LAYER)  # polygons
roads = gpd.read_file(ROADS_PATH, layer=ROAD_LAYER)  # lines
gnaf_df = pd.read_parquet(GNAF_PATH)

addr_cols = [
    c
    for c in gnaf_df.columns
    if any(
        k in c.lower() for k in ("address", "formatted", "full", "display", "street")
    )
]
addr = ensure_geometry_from_lonlat(gnaf_df)
addr["address_text"] = (
    gnaf_df[addr_cols[0]]
    if addr_cols
    else gnaf_df.get("address", gnaf_df.index.astype(str))
)

Using layers: cadastre / roads


project to local metric CRS


In [ ]:
addr_m, epsg = to_local_metric_crs(addr)
cad_m = cad.to_crs(epsg)
roads_m = roads.to_crs(epsg)
print("Working CRS EPSG:", epsg)

Working CRS EPSG: 32756


nearest joins


In [16]:
# Fixed version of the nearest joins with proper data alignment
nearest_parcel = gpd.sjoin_nearest(
    addr_m[["address_text", "geometry"]],
    cad_m[["geometry"]],
    how="left",
    distance_col="dist_to_parcel",
).rename(columns={"index_right": "parcel_idx"})

nearest_parcel = nearest_parcel.join(
    cad_m[["geometry"]], on="parcel_idx", rsuffix="_parcel"
)
nearest_parcel = gpd.GeoDataFrame(nearest_parcel, geometry="geometry")

parcel_centroids = nearest_parcel["geometry_parcel"].centroid
parcel_gdf = gpd.GeoDataFrame(
    nearest_parcel[["address_text"]], geometry=parcel_centroids, crs=epsg
)

parcel_to_road = gpd.sjoin_nearest(
    parcel_gdf, roads_m[["geometry"]], how="left", distance_col="parcel_road_dist"
).rename(columns={"index_right": "road_idx"})

# Merge the road information back to nearest_parcel using the index
# This ensures proper alignment and avoids length mismatch errors
nearest_parcel = nearest_parcel.merge(
    parcel_to_road[["road_idx", "parcel_road_dist"]],
    left_index=True,
    right_index=True,
    how="left",
)

print(f"✓ Successfully created nearest_parcel with {len(nearest_parcel)} rows")
print(f"Columns: {list(nearest_parcel.columns)}")

✓ Successfully created nearest_parcel with 617622 rows
Columns: ['address_text', 'geometry', 'parcel_idx', 'dist_to_parcel', 'geometry_parcel', 'road_idx', 'parcel_road_dist']


In [ ]:
# Calculate house orientations based on parcel geometry and nearest road
def calculate_house_orientation(parcel_geom, road_geom, parcel_centroid):
    """
    Calculate the orientation a house faces based on:
    1. The parcel geometry (property boundary)
    2. The nearest road geometry
    3. The parcel centroid (house location)

    Returns the compass direction the house faces (N, NE, E, SE, S, SW, W, NW)
    """
    if parcel_geom is None or road_geom is None or parcel_centroid is None:
        return "Unknown"

    try:
        # Get the parcel boundary edges
        parcel_edges = segmentize_polygon_edges(parcel_geom)
        if not parcel_edges:
            return "Unknown"

        # Find the edge closest to the road
        min_dist = float("inf")
        closest_edge = None

        for edge in parcel_edges:
            # Calculate distance from edge to road
            dist = edge.distance(road_geom)
            if dist < min_dist:
                min_dist = dist
                closest_edge = edge

        if closest_edge is None:
            return "Unknown"

        # Get the midpoint of the closest edge
        edge_midpoint = closest_edge.interpolate(0.5, normalized=True)

        # Calculate bearing from house (centroid) to the road-facing edge
        # This gives us the direction the house faces
        bearing = bearing_deg(parcel_centroid, edge_midpoint)

        # Convert bearing to compass direction
        return compass8(bearing)

    except Exception as e:
        print(f"Error calculating orientation: {e}")
        return "Unknown"


# Apply orientation calculation to our data
print("Calculating house orientations...")
orientations = []

for idx, row in nearest_parcel.iterrows():
    parcel_geom = row.get("geometry_parcel")
    road_idx = row.get("road_idx")

    if pd.isna(road_idx) or road_idx is None:
        orientations.append("Unknown")
        continue

    # Get the road geometry
    road_geom = roads_m.iloc[int(road_idx)]["geometry"]
    parcel_centroid = row["geometry"]

    orientation = calculate_house_orientation(parcel_geom, road_geom, parcel_centroid)
    orientations.append(orientation)

# Add orientations to the dataframe
nearest_parcel["orientation"] = orientations

print(f"✓ Calculated orientations for {len(nearest_parcel)} properties")
print(f"Orientation distribution:")
print(nearest_parcel["orientation"].value_counts())

In [ ]:
# Create the final output file with address and orientation
output_data = nearest_parcel[["address_text", "orientation"]].copy()

# Clean up the data
output_data = output_data.rename(columns={"address_text": "address"})

# Remove any rows with missing data
output_data = output_data.dropna()

# Sort by address for better organization
output_data = output_data.sort_values("address").reset_index(drop=True)

print(f"✓ Created output dataset with {len(output_data)} properties")
print(f"Columns: {list(output_data.columns)}")
print("\nFirst 10 rows:")
print(output_data.head(10))

print(f"\nOrientation summary:")
print(output_data["orientation"].value_counts().sort_index())

In [ ]:
# Save the output to a CSV file
output_file = "house_orientations.csv"
output_data.to_csv(output_file, index=False)

print(f"✓ Saved results to {output_file}")
print(f"File contains {len(output_data)} properties with their orientations")

# Also save as a Parquet file for better performance
output_parquet = "house_orientations.parquet"
output_data.to_parquet(output_parquet, index=False)

print(f"✓ Also saved as {output_parquet}")

# Display some statistics
print(f"\n📊 Summary Statistics:")
print(f"Total properties analyzed: {len(output_data)}")
print(
    f"Properties with known orientation: {len(output_data[output_data['orientation'] != 'Unknown'])}"
)
print(
    f"Properties with unknown orientation: {len(output_data[output_data['orientation'] == 'Unknown'])}"
)

# Show the most common orientations
print(f"\n🏠 Most common house orientations:")
orientation_counts = output_data["orientation"].value_counts()
for orientation, count in orientation_counts.head(8).items():
    percentage = (count / len(output_data)) * 100
    print(f"  {orientation}: {count} properties ({percentage:.1f}%)")

In [ ]:
# Create a visualization of the orientation results
import matplotlib.pyplot as plt

# Create a pie chart of orientations
plt.figure(figsize=(10, 8))
orientation_counts = output_data["orientation"].value_counts()

# Create the pie chart
colors = plt.cm.Set3(range(len(orientation_counts)))
wedges, texts, autotexts = plt.pie(
    orientation_counts.values,
    labels=orientation_counts.index,
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
)

plt.title("Distribution of House Orientations", fontsize=16, fontweight="bold")
plt.axis("equal")

# Add a legend
plt.legend(
    wedges,
    [f"{label}: {count}" for label, count in orientation_counts.items()],
    title="Orientations",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1),
)

plt.tight_layout()
plt.show()

# Create a bar chart as well
plt.figure(figsize=(12, 6))
orientation_counts.plot(kind="bar", color="skyblue", edgecolor="black")
plt.title("Number of Properties by Orientation", fontsize=14, fontweight="bold")
plt.xlabel("Orientation", fontsize=12)
plt.ylabel("Number of Properties", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)

# Add value labels on bars
for i, v in enumerate(orientation_counts.values):
    plt.text(i, v + 0.5, str(v), ha="center", va="bottom")

plt.tight_layout()
plt.show()